In [ ]:
import pandas as pd
# Loading the CSV file
df = pd.read_csv(
    "/Users/hd/Desktop/prompt-sensitivity-llms/src/outputs/responses_gemini_Gemini-2.0-Flash_5_20250808_1114.csv"
)

In [ ]:
# Step 1: structure sanity check
print(df.shape)
print(df.columns.tolist())

In [ ]:
errors = df["err"].unique().tolist()
print("Unique error messages:", errors)

In [ ]:
df["has_error"] = df["err"].notna()
df["has_error"].value_counts()

In [ ]:
# Filter only error rows, show err + response
df[df["has_error"]][["err", "response"]]

In [ ]:
# Breakdown of errors by domain
error_by_domain = (
    df[df["has_error"]].groupby("domain").size().sort_values(ascending=False)
)

# Breakdown of errors by variant
error_by_variant = (
    df[df["has_error"]].groupby("variant").size().sort_values(ascending=False)
)

# Combined breakdown (domain + variant)
error_combo = (
    df[df["has_error"]]
    .groupby(["domain", "variant"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

error_by_domain, error_by_variant, error_combo

In [ ]:
# Count errors per run
error_by_run = df[df["has_error"]].groupby("run_id").size()

# Count errors per run + domain
error_by_run_domain = (
    df[df["has_error"]].groupby(["run_id", "domain"]).size().reset_index(name="count")
)

error_by_run, error_by_run_domain.sort_values(
    ["run_id", "count"], ascending=[True, False]
)

In [ ]:
# --- Check how far each response is from the target of 100 words ---

# Calculate difference from target
df["word_diff"] = df["word_len"] - 100

# Summary statistics
df["word_diff"].describe()

# Optional: see top 5 most over/under the target
df.sort_values("word_diff").head(5)[["domain", "variant", "word_len", "word_diff"]]
df.sort_values("word_diff", ascending=False).head(5)[
    ["domain", "variant", "word_len", "word_diff"]
]

In [ ]:
# --- Average response time per variant and domain ---

# Mean latency per variant
df.groupby("variant")["latency_ms"].mean().sort_values()

# Mean latency per domain
df.groupby("domain")["latency_ms"].mean().sort_values()

In [ ]:
# --- Find responses that are much shorter than expected (possible cut-offs) ---

df[df["word_len"] < 50][["domain", "variant", "prompt", "word_len", "response"]]

In [ ]:
# --- Check if different prompts produced exactly the same response ---

duplicate_count = df.duplicated(subset=["response"]).sum()
print(f"Number of duplicate responses: {duplicate_count}")

# Optional: list a few duplicate examples
df[df.duplicated(subset=["response"], keep=False)].sort_values("response").head(10)[
    ["domain", "variant", "prompt", "response"]
]